In [ ]:
!pip install -U transformers dataset accelerate

  Using cached dataset-2.0.0-py3-none-any.whl.metadata (3.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 11.5 MB/s eta 0:00:00


In [ ]:
import torch
print("GPU available:",torch.cuda.is_available())
if torch.cuda.is_available():
  print("GPU:",torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import Dataset

# 1. Installing required libraries (already done in a previous cell, but good to have a reminder)
# !pip install -U transformers datasets accelerate

print("Libraries loaded successfully!")

# 2. Loading GPT-2 and its tokenizer
# We'll use the smaller gpt2 model for faster fine-tuning on a T4 GPU for demonstration purposes.
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# GPT-2 tokenizer does not have a padding token by default, which is needed for training.
# We set it to the `eos_token` (end-of-sequence token).
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

print(f"Loaded model: {model_name}")
print(f"Tokenizer vocabulary size: {len(tokenizer)}")

# 3. Creating/loading a custom dataset
# For this example, we'll create a small synthetic dataset.
# In a real scenario, you would load your text data from a file (e.g., .txt, .csv).

raw_text_data = [
    "Generative AI is a fascinating field that involves creating new content.",
    "Large Language Models (LLMs) like GPT-2 are at the forefront of AI text generation.",
    "Fine-tuning a pre-trained model allows it to specialize in specific tasks or styles.",
    "Google Colab provides free GPU access, which is excellent for AI development.",
    "This internship task focuses on making GPT-2 generate coherent and relevant text.",
    "The future of AI is exciting, with new advancements emerging constantly.",
    "Text generation can be used for creative writing, summarization, and chatbots."
]

# Convert the list of texts into a Hugging Face Dataset object
dataset = Dataset.from_dict({"text": raw_text_data})

print(f"Created dataset with {len(dataset)} examples.")
print("Sample data:", dataset[0]['text'])

# 4. Tokenizing the dataset
# Define a function to tokenize the text data.
# We set truncation=True to handle texts longer than the model's max input length.
# We set `max_length` to a reasonable value for GPT-2, which is 1024.
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

print("Dataset tokenized successfully!")
print("Sample tokenized data (input_ids):", tokenized_dataset[0]['input_ids'])

# 5. Fine-tuning GPT-2

# Define training arguments
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned", # Directory to save checkpoints and logs
    overwrite_output_dir=True,    # Overwrite the content of the output directory
    num_train_epochs=3,           # Number of training epochs
    per_device_train_batch_size=2, # Batch size for training per GPU/CPU
    save_steps=10_000,            # Save checkpoint every X updates steps
    save_total_limit=2,           # Only keep the last 2 checkpoints
    logging_dir="./logs",         # Directory for storing logs
    logging_steps=50,             # Log every X updates steps
    gradient_accumulation_steps=4, # Accumulate gradients over 4 steps to simulate a larger batch size
    learning_rate=5e-5,           # Learning rate
    weight_decay=0.01,            # Weight decay
    fp16=torch.cuda.is_available() # Use mixed precision training if GPU is available
)

# Data collator for language modeling (will group samples into batches and apply padding)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# Start training
print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning completed!")

# 6. Saving the fine-tuned model
output_dir = "./fine_tuned_gpt2"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Fine-tuned model and tokenizer saved to {output_dir}/")

# 7. Loading the trained model
loaded_tokenizer = AutoTokenizer.from_pretrained(output_dir)
loaded_model = AutoModelForCausalLM.from_pretrained(output_dir)
print("Fine-tuned model and tokenizer loaded successfully!")

# Move model to GPU if available
if torch.cuda.is_available():
    loaded_model.to('cuda')
    print("Model moved to GPU.")

# 8. Generating text from custom prompts
# 9. Showing sample outputs

def generate_text(model, tokenizer, prompt, max_length=100, num_return_sequences=1, temperature=0.7, top_k=50, top_p=0.95):
    input_ids = tokenizer.encode(prompt, return_tensors='pt')
    if torch.cuda.is_available():
        input_ids = input_ids.to('cuda')

    # Generate text using the fine-tuned model
    output = model.generate(
        input_ids,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        no_repeat_ngram_size=2, # Avoid repeating n-grams
        do_sample=True,         # Use sampling for more diverse text
        temperature=temperature, # Control randomness
        top_k=top_k,            # Filter out low probability words
        top_p=top_p,            # Filter out words by cumulative probability
        pad_token_id=tokenizer.eos_token_id # Ensure pad_token_id is set for generation
    )

    generated_texts = [tokenizer.decode(output_ids, skip_special_tokens=True) for output_ids in output]
    return generated_texts

print("\n--- Generated Text Samples ---")
prompts = [
    "Generative AI will revolutionize",
    "The key to successful text generation is",
    "My internship task is to fine-tune GPT-2 for"
]

for i, prompt in enumerate(prompts):
    print(f"\nPrompt {i+1}: '{prompt}'")
    generated_output = generate_text(loaded_model, loaded_tokenizer, prompt, max_length=50, num_return_sequences=1)
    for text in generated_output:
        print(f"Generated: {text}")
